# 🛡️ Agent Engineering Challenge

**Duration:** 90 minutes &nbsp;|&nbsp; **Teams of 3–4** &nbsp;|&nbsp; **Claude Messages API or Agent SDK**

---

## What You're Building

An AI agent that automates RFP (Request for Proposal) responses for a cybersecurity vendor. Your agent will:

1. **Parse** a questionnaire into categorized questions
2. **Retrieve** relevant source material from a knowledge base
3. **Draft** polished, cited answers
4. **Review** answers for cross-question consistency
5. **Export** structured JSON output

At the end, your team demos the agent live and explains your design decisions.

### How You'll Be Evaluated

| Criterion | Weight | What "Excellent" Looks Like |
|-----------|--------|----------------------------|
| **Agent Quality** | 60% | Clean tool design, effective prompts, structured JSON output, handles edge cases, review step catches inconsistencies |
| **Demo Clarity** | 40% | Clear architecture walkthrough, live run on a fresh RFP, honest retrospective on tradeoffs |

**Stretch goal (unscored but impressive):** Build eval assertions that validate your agent's output quality.

### Suggested Time Allocation

| Phase | Duration | Focus |
|-------|----------|-------|
| Setup & Planning | 5 min | Verify API, assign roles, sketch architecture |
| Build | 65 min | Agent loop, prompt design, testing |
| Demo Prep | 5 min | Prepare your walkthrough |
| Demos | 15 min | Live presentations (3 min per team) |

---

## Part 0: Environment Setup

Run the cells below to install dependencies and verify your API connection.

In [ ]:
# Install dependencies
!pip install anthropic --upgrade -q

print("✓ Dependencies installed")

In [ ]:
import os
import json
from typing import Optional

import anthropic

# === API KEY SETUP ===
ANTHROPIC_API_KEY = ""  # <-- Paste your Anthropic API key here

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY not found.\n"
        "Set it via: export ANTHROPIC_API_KEY='your-key-here'\n"
        "Or paste directly in the cell above."
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✓ Anthropic client initialized")

In [ ]:
# Verify API connectivity
try:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=20,
        messages=[{"role": "user", "content": "Say 'ready' and nothing else."}]
    )
    print(f"✓ API connection verified: {response.content[0].text}")
    print(f"  Model: {response.model}")
except Exception as e:
    print(f"❌ API connection failed: {e}")

---

## Part 1: The Brief

### The Situation

**Helios Security** is a mid-market cybersecurity vendor selling endpoint protection, SIEM, and managed detection & response. Their sales team responds to **40+ RFPs per quarter**, each containing 50–200 questions spanning technical architecture, compliance certifications, pricing models, and company background.

Today, each RFP takes a solutions engineer **6–8 hours** to complete. The process is manual: hunt through a Confluence wiki of past proposals, cross-reference product documentation, and copy-paste answers into a spreadsheet — adjusting tone and specificity for each prospect. Answers frequently **contradict each other** across questions because no one reviews the full response holistically.

Helios wants an AI agent that takes in an RFP questionnaire and produces a draft response in minutes — decomposing questions, retrieving relevant source material, synthesizing polished answers, and flagging anything the human needs to review.

**The goal:** Cut first-draft time from 8 hours to under 15 minutes and eliminate cross-answer inconsistencies.

### The Agent Pipeline

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  PARSE   │───▶│ RETRIEVE │───▶│  DRAFT   │───▶│  REVIEW  │───▶│  EXPORT  │
│          │    │          │    │          │    │          │    │          │
│ Break    │    │ Search   │    │ Generate │    │ Check    │    │ Return   │
│ into Qs  │    │ mock KB  │    │ answers  │    │ for      │    │ struct.  │
│ + tag    │    │ via tool │    │ w/ cites │    │ contra-  │    │ JSON     │
│ category │    │          │    │          │    │ dictions │    │          │
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

### Sample RFP Questions

| ID | Category | Question |
|----|----------|----------|
| Q1 | Technical | Describe your platform's approach to real-time threat detection. What data sources are ingested, and what is the average detection-to-alert latency? |
| Q2 | Compliance | List all compliance certifications your organization currently holds (SOC 2, ISO 27001, FedRAMP, etc.) and the date of most recent audit for each. |
| Q3 | Pricing | Provide per-seat pricing for 500, 1,000, and 5,000 endpoints. Are volume discounts available? Is there a minimum contract term? |
| Q4 | Company-Info | How many customers do you currently serve in the financial services vertical? Provide 2–3 reference accounts. |
| Q5 | Technical + Compliance | How does your platform handle data residency requirements for customers operating in the EU? Describe encryption at rest and in transit. |

---

## Part 2: Mock Knowledge Base (Pre-Built)

Your agent needs data to retrieve from. Below is a pre-built mock knowledge base with realistic Helios Security content. **You can extend this with additional entries**, but the baseline is ready to use.

In [ ]:
# ============================================================
# MOCK KNOWLEDGE BASE
# Pre-populated with realistic Helios Security content.
# Extend with additional entries if you want richer agent behavior.
# ============================================================

KNOWLEDGE_BASE = {
    "threat_detection": {
        "source": "Helios Platform Architecture Doc v4.2",
        "content": (
            "Helios Sentinel uses a multi-layered detection engine combining "
            "signature-based matching, behavioral analysis, and ML-driven anomaly detection. "
            "Data sources include endpoint telemetry (process events, file system changes, "
            "network connections), cloud workload logs (AWS CloudTrail, Azure Activity Log, "
            "GCP Audit Log), network flow data (NetFlow v9/IPFIX), and email gateway events. "
            "Average detection-to-alert latency is 2.3 seconds for signature matches and "
            "18 seconds for behavioral detections. Our SIEM correlation engine processes "
            "up to 50,000 events per second per tenant."
        ),
        "tags": ["technical", "detection", "latency", "architecture"]
    },
    "compliance_certs": {
        "source": "Helios Compliance & Certifications Register 2025",
        "content": (
            "Current certifications: SOC 2 Type II (audited December 2024 by Deloitte), "
            "ISO 27001:2022 (certified March 2024 by BSI), FedRAMP Moderate (authorized "
            "June 2024, sponsored by DHS), HIPAA (BAA available, last assessment October 2024), "
            "PCI DSS v4.0 Level 1 Service Provider (validated September 2024 by Coalfire). "
            "StateRAMP authorized (January 2025). All certifications maintained on continuous "
            "monitoring basis with quarterly internal audits."
        ),
        "tags": ["compliance", "certifications", "audit", "soc2", "fedramp"]
    },
    "pricing_model": {
        "source": "Helios Commercial Pricing Sheet Q1 2025",
        "content": (
            "Endpoint Protection Platform (EPP+EDR bundle): "
            "500 endpoints: $18/seat/month ($108,000/year). "
            "1,000 endpoints: $15/seat/month ($180,000/year) — 17% volume discount. "
            "5,000 endpoints: $11/seat/month ($660,000/year) — 39% volume discount. "
            "Minimum contract term: 12 months. Multi-year discounts: 2-year = additional 5%, "
            "3-year = additional 10%. SIEM add-on: +$6/seat/month. "
            "MDR add-on: +$12/seat/month. All pricing excludes professional services."
        ),
        "tags": ["pricing", "commercial", "discount", "contract"]
    },
    "financial_services_customers": {
        "source": "Helios Customer Success — Vertical Report 2024",
        "content": (
            "Helios currently serves 47 customers in financial services, including "
            "12 banks, 8 insurance carriers, 15 asset management firms, and 12 fintech companies. "
            "Reference accounts (approved for external use): "
            "1) Meridian National Bank — 3,200 endpoints, EPP+EDR+SIEM, deployed since 2022. "
            "2) Crestview Capital Partners — 850 endpoints, EPP+MDR, deployed since 2023. "
            "3) Apex Insurance Group — 5,100 endpoints, full platform, deployed since 2021. "
            "Average NPS in financial services vertical: 72."
        ),
        "tags": ["company-info", "customers", "financial-services", "references"]
    },
    "data_residency_eu": {
        "source": "Helios Data Sovereignty & Privacy Whitepaper v3.1",
        "content": (
            "Helios supports full EU data residency through dedicated infrastructure in "
            "Frankfurt (AWS eu-central-1) and Dublin (AWS eu-west-1). Customer data never "
            "leaves the selected region. Encryption at rest: AES-256-GCM with customer-managed "
            "keys (AWS KMS or BYOK). Encryption in transit: TLS 1.3 for all API and agent "
            "communications, with certificate pinning for endpoint agents. "
            "GDPR Data Processing Agreement (DPA) included in all EU contracts. "
            "Annual third-party penetration testing by NCC Group. "
            "Data retention: configurable per tenant, default 90 days for raw telemetry, "
            "13 months for aggregated alerts."
        ),
        "tags": ["technical", "compliance", "data-residency", "eu", "encryption", "gdpr"]
    },
    "past_rfp_detection_answer": {
        "source": "Acme Corp RFP Response — March 2024",
        "content": (
            "Q: Describe your real-time threat detection capabilities. "
            "A: Helios Sentinel provides sub-3-second detection for known threat patterns "
            "and under 20 seconds for behavioral anomalies. Our detection engine ingests "
            "endpoint telemetry, network flows, cloud audit logs, and email events. "
            "The SIEM correlation engine handles 50K EPS per tenant. "
            "We maintain a 99.7% true positive rate on our top 100 detection rules, "
            "validated quarterly against MITRE ATT&CK framework."
        ),
        "tags": ["technical", "detection", "past-rfp"]
    },
    "past_rfp_compliance_answer": {
        "source": "NovaTech RFP Response — July 2024",
        "content": (
            "Q: What compliance certifications do you hold? "
            "A: Helios holds SOC 2 Type II, ISO 27001, FedRAMP Moderate, PCI DSS v4.0, "
            "and HIPAA compliance. All certifications are actively maintained with "
            "continuous monitoring. We provide audit reports upon request under NDA. "
            "Our security team of 14 full-time engineers manages compliance programs."
        ),
        "tags": ["compliance", "certifications", "past-rfp"]
    }
}


def search_knowledge_base(query: str, category: Optional[str] = None) -> list[dict]:
    """
    Search the mock knowledge base.
    Returns matching entries ranked by keyword overlap.
    """
    query_terms = set(query.lower().split())
    results = []

    for entry_id, entry in KNOWLEDGE_BASE.items():
        # Score by keyword overlap with content + tags
        entry_text = (entry["content"] + " " + " ".join(entry["tags"])).lower()
        overlap = len(query_terms & set(entry_text.split()))

        # Boost if category matches a tag
        if category and category.lower() in [t.lower() for t in entry["tags"]]:
            overlap += 5

        if overlap > 0:
            results.append({
                "id": entry_id,
                "source": entry["source"],
                "content": entry["content"],
                "relevance_score": overlap
            })

    # Sort by relevance, return top 3
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results[:3]


# Quick test
test_results = search_knowledge_base("threat detection latency", category="technical")
print(f"✓ Knowledge base loaded ({len(KNOWLEDGE_BASE)} entries)")
print(f"  Test search 'threat detection latency': {len(test_results)} results")
print(f"  Top result: {test_results[0]['source']}")

---

## Part 3: Tool Definition

Define the `search_kb` tool so Claude can retrieve information from the knowledge base during its agent loop.

In [ ]:
# Tool definition for the Claude API
SEARCH_KB_TOOL = {
    "name": "search_kb",
    "description": (
        "Search the Helios Security knowledge base for information relevant to "
        "answering an RFP question. Returns up to 3 matching documents with source "
        "attribution. Use this to find product docs, past proposal answers, compliance "
        "records, and pricing information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use keywords from the RFP question"
            },
            "category": {
                "type": "string",
                "enum": ["technical", "compliance", "pricing", "company-info"],
                "description": "Optional category filter to narrow results"
            }
        },
        "required": ["query"]
    }
}


def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call and return the result as a string."""
    if tool_name == "search_kb":
        results = search_knowledge_base(
            query=tool_input["query"],
            category=tool_input.get("category")
        )
        return json.dumps(results, indent=2)
    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})


print("✓ Tool definition ready")

---

## Part 4: Level 0 Agent (Working Baseline)

Below is a working agent that handles a **single question** end-to-end. It:
- Sends the question to Claude with the `search_kb` tool available
- Handles the tool use loop (Claude calls the tool → we execute it → send results back)
- Returns a structured answer

**Run this to confirm the agent loop works**, then extend it in Part 5.

In [ ]:
SYSTEM_PROMPT = """You are an AI assistant helping Helios Security respond to RFP questionnaires.

For each question, you must:
1. Use the search_kb tool to find relevant source material
2. Draft a professional, detailed answer grounded in the retrieved sources
3. Cite your sources by name
4. If the knowledge base doesn't contain enough information, flag the answer as low-confidence

Return your answer as JSON with this structure:
{
    "question_id": "Q1",
    "category": "technical",
    "answer": "Your drafted answer here...",
    "sources": ["Source Name 1", "Source Name 2"],
    "confidence": "high" | "medium" | "low",
    "flags": ["any concerns or notes for human review"]
}

Be specific, professional, and concise. Use concrete numbers from the source material."""


def answer_single_question(
    question_id: str,
    question_text: str,
    category: str,
    model: str = "claude-sonnet-4-20250514",
) -> dict:
    """
    Level 0 agent: answers a single RFP question with tool use.
    Handles the full tool use loop.
    """
    messages = [
        {
            "role": "user",
            "content": (
                f"Answer this RFP question.\n\n"
                f"Question ID: {question_id}\n"
                f"Category: {category}\n"
                f"Question: {question_text}\n\n"
                f"Search the knowledge base for relevant information, then draft your answer."
            )
        }
    ]

    # Agent loop: keep going until Claude stops calling tools
    max_turns = 5  # Safety limit
    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=messages,
            tools=[SEARCH_KB_TOOL],
        )

        # If Claude is done (no more tool calls), extract the answer
        if response.stop_reason == "end_turn":
            # Find the text block with the JSON answer
            for block in response.content:
                if block.type == "text":
                    try:
                        # Try to parse JSON from the response
                        text = block.text
                        # Handle markdown code blocks
                        if "```json" in text:
                            text = text.split("```json")[1].split("```")[0]
                        elif "```" in text:
                            text = text.split("```")[1].split("```")[0]
                        return json.loads(text.strip())
                    except json.JSONDecodeError:
                        return {"raw_response": block.text, "parse_error": True}
            break

        # If Claude wants to use a tool, execute it and continue
        if response.stop_reason == "tool_use":
            # Add assistant's response to message history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool call and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = handle_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            messages.append({"role": "user", "content": tool_results})

    return {"error": "Max turns reached without completing"}


print("✓ Level 0 agent defined")

In [ ]:
# Test: Answer Q1
print("Answering Q1 (threat detection)...")
result = answer_single_question(
    question_id="Q1",
    question_text=(
        "Describe your platform's approach to real-time threat detection. "
        "What data sources are ingested, and what is the average detection-to-alert latency?"
    ),
    category="technical",
)

print(json.dumps(result, indent=2))

---

## Part 5: 🛠️ YOUR TASK — Multi-Question Agent

The Level 0 agent handles one question at a time. Your job is to build a **full pipeline** that:

1. **Accepts a list of RFP questions** (the full questionnaire)
2. **Processes each question** through the agent loop
3. **Collects all answers** into a structured response
4. **Exports as JSON** with all required fields

### Approaches to Consider

- **Sequential:** Process questions one by one using `answer_single_question()`. Simple, reliable, but slower.
- **Batch prompt:** Send all questions to Claude in a single prompt and let it call the tool multiple times. Faster, but harder to control.
- **Hybrid:** Parse and categorize first, then process by category to reuse retrieved context.

### Starter Questions

In [ ]:
# RFP Questionnaire
RFP_QUESTIONS = [
    {
        "id": "Q1",
        "category": "technical",
        "text": (
            "Describe your platform's approach to real-time threat detection. "
            "What data sources are ingested, and what is the average detection-to-alert latency?"
        ),
    },
    {
        "id": "Q2",
        "category": "compliance",
        "text": (
            "List all compliance certifications your organization currently holds "
            "(SOC 2, ISO 27001, FedRAMP, etc.) and the date of most recent audit for each."
        ),
    },
    {
        "id": "Q3",
        "category": "pricing",
        "text": (
            "Provide per-seat pricing for 500, 1,000, and 5,000 endpoints. "
            "Are volume discounts available? Is there a minimum contract term?"
        ),
    },
    {
        "id": "Q4",
        "category": "company-info",
        "text": (
            "How many customers do you currently serve in the financial services vertical? "
            "Provide 2–3 reference accounts."
        ),
    },
    {
        "id": "Q5",
        "category": "technical",
        "text": (
            "How does your platform handle data residency requirements for customers "
            "operating in the EU? Describe encryption at rest and in transit."
        ),
    },
]

In [ ]:
# ============================================================
# 🛠️ YOUR CODE: Build the multi-question agent
# ============================================================
#
# Process all RFP_QUESTIONS and collect structured responses.
#
# Your output should be a list of answer objects, each containing:
#   - question_id
#   - category
#   - answer
#   - sources
#   - confidence (high / medium / low)
#   - flags (list of concerns for human review)
#
# HINT: The simplest approach is a loop over answer_single_question().
# More advanced: modify the system prompt to handle batches.
# ============================================================

def process_rfp(questions: list[dict]) -> list[dict]:
    """Process a full RFP questionnaire and return structured answers."""
    answers = []

    # TODO: Your implementation here
    # Option 1 (simple): Loop through questions, call answer_single_question for each
    # Option 2 (advanced): Send all questions in one prompt with batch processing

    pass  # Replace with your implementation

    return answers


# Run it
print("Processing RFP...")
all_answers = process_rfp(RFP_QUESTIONS)

# Display results
if all_answers:
    for ans in all_answers:
        q_id = ans.get('question_id', '?')
        conf = ans.get('confidence', '?')
        print(f"  {q_id}: confidence={conf}, sources={len(ans.get('sources', []))}")
    print(f"\n✓ Processed {len(all_answers)} questions")
else:
    print("❌ No answers returned — implement process_rfp() above")

---

## Part 6: 🛠️ YOUR TASK — Consistency Review Step

The customer specifically asked for **cross-answer consistency checking**. Answers drafted individually often contradict each other (e.g., Q2 says "FedRAMP authorized June 2024" but Q5 says "FedRAMP certified in 2023").

Build a review step that:
1. Takes all drafted answers as input
2. Identifies potential contradictions or inconsistencies
3. Flags issues for human review

**This is what separates a good agent from a great one.** In real deployments, consistency is the #1 quality issue in automated RFP responses.

In [ ]:
# ============================================================
# 🛠️ YOUR CODE: Consistency review step
# ============================================================
#
# Send all answers to Claude for cross-answer review.
# Look for: contradictions, inconsistent data points, tone mismatches.
#
# HINT: Format all answers as a single prompt and ask Claude to
# identify any inconsistencies. No tool use needed for this step.
# ============================================================

def review_answers(answers: list[dict]) -> dict:
    """
    Review all drafted answers for cross-question consistency.
    Returns a review report with any flagged issues.
    """

    # TODO: Your implementation here
    # 1. Format all answers into a single prompt
    # 2. Ask Claude to identify contradictions, inconsistencies, tone issues
    # 3. Return a structured review report

    pass  # Replace with your implementation


# Run review on your answers
if all_answers:
    print("Running consistency review...")
    review = review_answers(all_answers)
    print(json.dumps(review, indent=2))
else:
    print("⚠️ No answers to review yet — complete Part 5 first")

---

## Part 7: Export Final Output

Package your agent's output as clean JSON — the deliverable Helios would actually receive.

In [ ]:
# Build the final export
final_output = {
    "rfp_name": "Sample RFP — Agent Engineering Challenge",
    "total_questions": len(RFP_QUESTIONS),
    "answers": all_answers if all_answers else [],
    "review": review if 'review' in dir() and review else "Review not completed",
    "metadata": {
        "model": "claude-sonnet-4-20250514",
        "knowledge_base_entries": len(KNOWLEDGE_BASE),
    }
}

print(json.dumps(final_output, indent=2))

---

## Part 8: ⭐ STRETCH — Eval Assertions

*This section is optional and unscored, but demonstrates rigor.*

Build assertions that validate your agent's output quality. Good evals test real failure modes, not just happy paths.

### What to Test

| Dimension | Example Assertion |
|-----------|-------------------|
| **Accuracy** | Q3 pricing answer contains "$18/seat/month" for 500 endpoints |
| **Source attribution** | Every answer has at least one source cited |
| **Consistency** | Q2 and Q5 don't contradict each other on certification dates |
| **Confidence calibration** | Questions with no KB match are flagged as low-confidence |
| **Edge cases** | What happens with an ambiguous or unanswerable question? |

In [ ]:
# ============================================================
# ⭐ STRETCH: Write eval assertions
# ============================================================

def run_evals(answers: list[dict]) -> dict:
    """Run quality assertions against agent output."""
    results = {"passed": 0, "failed": 0, "details": []}

    for ans in answers:
        # Example assertion: every answer has sources
        has_sources = len(ans.get("sources", [])) > 0
        results["details"].append({
            "question": ans.get("question_id"),
            "assertion": "has_sources",
            "passed": has_sources,
        })
        if has_sources:
            results["passed"] += 1
        else:
            results["failed"] += 1

    # TODO: Add more assertions
    # - Check that pricing answer contains specific numbers
    # - Check that confidence is a valid value
    # - Check for cross-answer consistency
    # - Test an edge case question (e.g., a question with no KB match)

    return results


if all_answers:
    eval_results = run_evals(all_answers)
    print(f"Eval Results: {eval_results['passed']} passed, {eval_results['failed']} failed")
    for detail in eval_results["details"]:
        status = "✓" if detail["passed"] else "❌"
        print(f"  {status} {detail['question']}: {detail['assertion']}")
else:
    print("⚠️ Complete Part 5 first")

---

## 🏁 Demo Prep

Your team has **3 minutes** to present. Suggested structure:

| Segment | Duration | Content |
|---------|----------|--------|
| Architecture | ~45 sec | How you structured the agent, what design decisions you made |
| Live run | ~1 min | Run your agent on the RFP, walk through the output |
| Retrospective | ~1 min | What worked, what broke, what you'd improve with more time |

### Questions to answer in your demo:
1. How did you handle the tool use loop? Sequential or batch?
2. What does your review step actually catch?
3. What would break if you deployed this to production? What's missing?
4. If you had another hour, what would you build first?